In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_muon.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_all.df"
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_data.df"
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_5e18_muon_data.df"

mc_bnb_df = load_df(bnb_path, keys2load, 100)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
print(mc_bnb_pfp_df.columns)

In [ ]:
print(mc_bnb_pfp_df.index)

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

if "pion" in bnb_path:
    data_tot_pot = 8.371e+19
else:
    data_tot_pot = 5.948e+18
    
print("data_tot_pot: %.3e" %(data_tot_pot))
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_pfp_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_pfp_df))

In [ ]:
import pandas as pd

# Define the target 4 index levels identifying unique tracks/slices
target_levels = ['__ntuple', 'entry', 'rec.slc..index', 'rec.slc.reco.pfp..index']

# --- 1. Combined Unique Combinations Across hit0 + hit1 + hit2 ---
hit_dfs = {
    'hit0': mc_bnb_hit0_df,
    'hit1': mc_bnb_hit1_df,
    'hit2': mc_bnb_hit2_df,
}

hit_tuple_sets = []
total_hit_rows = 0

for name, df in hit_dfs.items():
    if not df.empty:
        total_hit_rows += len(df)
        # Extract target 4-tuples as a set
        tuples_set = set(
            df.index.to_frame()[target_levels].itertuples(index=False, name=None)
        )
        hit_tuple_sets.append(tuples_set)

# Union of unique 4-tuples across hit0, hit1, and hit2
if hit_tuple_sets:
    combined_hit_unique_tuples = set.union(*hit_tuple_sets)
    n_unique_combined_hits = len(combined_hit_unique_tuples)
else:
    n_unique_combined_hits = 0

print(f"Combined (hit0+hit1+hit2) total hit rows: {total_hit_rows}")
print(f"Combined (hit0+hit1+hit2) unique 4-tuple combinations: {n_unique_combined_hits}")

print("\n" + "=" * 50 + "\n")

# --- 2. Unique Combinations for mc_bnb_pfp_df ---
mc_bnb_pfp_df = mc_bnb_pfp_df.sort_index(level='__ntuple', ascending=True)

pfp_tuples_set = set(
    mc_bnb_pfp_df.index.to_frame()[target_levels].itertuples(index=False, name=None)
)
n_unique_pfp = len(pfp_tuples_set)

print(f"mc_bnb_pfp_df total rows: {len(mc_bnb_pfp_df)}")
print(f"mc_bnb_pfp_df unique 4-tuple combinations: {n_unique_pfp}")

# --- Optional Check: Overlap between Hits and PFP ---
if n_unique_combined_hits > 0 and n_unique_pfp > 0:
    overlap = len(combined_hit_unique_tuples.intersection(pfp_tuples_set))
    print("\n" + "=" * 50 + "\n")
    print(f"Unique tracks present in BOTH PFP and Hits: {overlap}")

In [ ]:
import pandas as pd

# Define your columns
p_type_col = ('pfp', 'trk', 'truth', 'p', 'p_type', '')
weight_col = ('slc', 'wgt', '', '', '', '')  # Optional: set to None if unweighted

# --- 1. Extract and Clean Data ---
valid_mask = mc_bnb_pfp_df[p_type_col].notna()
df_clean = mc_bnb_pfp_df[valid_mask]

if weight_col and weight_col in df_clean.columns:
    weights = df_clean[weight_col].fillna(1.0)
else:
    weights = pd.Series(1.0, index=df_clean.index)

# --- 2. Calculate Weighted & Unweighted Statistics ---
stats_df = pd.DataFrame({
    'p_type': df_clean[p_type_col],
    'weight': weights
})

summary = stats_df.groupby('p_type').agg(
    Counts=('weight', 'count'),
    Weighted_Yield=('weight', 'sum')
).reset_index()

total_counts = summary['Counts'].sum()
total_weighted = summary['Weighted_Yield'].sum()

summary['Raw_%'] = (summary['Counts'] / total_counts) * 100
summary['Weighted_%'] = (summary['Weighted_Yield'] / total_weighted) * 100

# Sort by weighted yield (descending)
summary = summary.sort_values(by='Weighted_Yield', ascending=False)

# --- 3. Pretty Print Output ---
print("\n" + "="*60)
print(f"{'p_type':<15} | {'Counts':<8} | {'Raw %':<8} | {'Weighted':<10} | {'Weighted %':<10}")
print("-" * 60)

for _, row in summary.iterrows():
    print(f"{str(row['p_type']):<15} | {int(row['Counts']):<8d} | {row['Raw_%']:<7.2f}% | {row['Weighted_Yield']:<10.1f} | {row['Weighted_%']:<9.2f}%")

print("-" * 60)
print(f"{'Total':<15} | {total_counts:<8d} | {100.0:<7.2f}% | {total_weighted:<10.1f} | {100.0:<9.2f}%")
print("="*60 + "\n")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_category_histogram(
    df: pd.DataFrame,
    var_col: tuple,
    category_col: tuple,
    weight_col: tuple = None,
    bins: np.ndarray = None,
    category_colors: dict = None,
    category_labels: dict = None,
    xlabel: str = "Reconstructed Variable",
    ylabel: str = "A.U.",
    title: str = None,
    vlines: tuple = None,  # 👈 Vertical lines support (e.g. (x1, x2) or None)
    stacked: bool = True,
    alpha: float = 0.3,
    linewidth: float = 1.8,
    figsize: tuple = (8, 6),
):
    # --- 1. Pre-process and Extract Data ---
    valid_mask = df[var_col].notna() & df[category_col].notna()
    plot_df = df[valid_mask].copy()

    if plot_df.empty:
        raise ValueError(
            "DataFrame has no valid rows after dropping NaNs in var_col and category_col."
        )

    # Resolve Weights
    if weight_col is not None and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0)
    else:
        weights = pd.Series(1.0, index=plot_df.index)

    # Determine Bins
    if bins is None:
        val_min, val_max = plot_df[var_col].min(), plot_df[var_col].max()
        bins = np.linspace(val_min, val_max, 51)

    # --- 2. Categorization & Sorting ---
    present_categories = plot_df[category_col].unique()

    # Sort categories by total weighted yield (largest first)
    cat_yields = [
        (cat, weights[plot_df[category_col] == cat].sum())
        for cat in present_categories
    ]
    sorted_categories = [
        cat for cat, _ in sorted(cat_yields, key=lambda x: x[1], reverse=True)
    ]

    # Prepare stacked datasets and weights
    grouped_data = [
        plot_df.loc[plot_df[category_col] == cat, var_col]
        for cat in sorted_categories
    ]
    grouped_weights = [
        weights[plot_df[category_col] == cat] for cat in sorted_categories
    ]

    # Defaults for colors & labels
    category_colors = category_colors or {}
    category_labels = category_labels or {}

    colors = [category_colors.get(cat, "#7f7f7f") for cat in sorted_categories]
    labels = [category_labels.get(cat, str(cat)) for cat in sorted_categories]

    # --- 3. Plotting ---
    fig, ax = plt.subplots(figsize=figsize)

    # Layer 1: Filled steps with opacity
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="stepfilled",
        color=colors,
        alpha=alpha,
        label=labels,
    )

    # Layer 2: Sharp outlines on step edges matching the category color
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="step",
        color=colors,
        linewidth=linewidth,
    )

    # --- Layer 3: Vertical Lines ---
    if vlines is not None:
        if isinstance(vlines, (int, float)):
            vlines = [vlines]
        for v in vlines:
            if v is not None:
                ax.axvline(
                    x=v,
                    color="gray",
                    linestyle="--",
                    linewidth=2.0,
                    zorder=4,
                )

    # --- 4. Styling & Formatting ---
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)
    if title:
        ax.set_title(title, fontsize=15, pad=12)

    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)  # Leave room for legend

    ax.tick_params(axis="both", which="both", labelsize=12, direction="in")
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    # Legend ordered according to the stack order and anchored to top-right
    ax.legend(
        loc="upper right",  # 👈 Forces top-right placement
        fontsize=12,
        frameon=True,
        framealpha=1.0,
        edgecolor="black",
        fancybox=False,
    )

    plt.tight_layout()
    return fig, ax

In [ ]:
# Setup column definitions matching your MultiIndex format
p_pion_col = ("pfp", "trk", "rangeP", "p_muon", "", "")
p_type_col = ("pfp", "trk", "truth", "p", "p_type", "")
weight_col = ("slc", "wgt", "", "", "", "")

p_type_labels = {
    "muon": r"$\mu$",
    "pion": r"$\pi^\pm$",
    "proton": r"$p$",
    "other": "Other / Unmatched",
}

bins = np.linspace(0.05, 2, 51)

# Execute plot call
fig, ax = plot_category_histogram(
    df=mc_bnb_pfp_df,
    var_col=p_pion_col,
    category_col=p_type_col,
    weight_col=weight_col,
    bins=bins,
    category_colors=category_colors_pfp,
    category_labels=p_type_labels,
    xlabel=r"Range-based $p_\mu$ [GeV]",
    ylabel="Weighted events",
    title=r"Range-based $p_\mu$ by true particle type",
    stacked=True,
)

plt.show()

In [ ]:

import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Define coordinate tuples matching MultiIndex schema
start_cols = {
    "x": ("pfp", "trk", "start", "x", "", ""),
    "y": ("pfp", "trk", "start", "y", "", ""),
    "z": ("pfp", "trk", "start", "z", "", ""),
}

end_cols = {
    "x": ("pfp", "trk", "end", "x", "", ""),
    "y": ("pfp", "trk", "end", "y", "", ""),
    "z": ("pfp", "trk", "end", "z", "", ""),
}

# Coordinate bin definitions
BINS_Y = np.linspace(-200, 200, 41)  # Vertical Axis (Y)
BINS_Z = np.linspace(0, 500, 51)     # Horizontal Axis (Z)


# --- 1. Define custom sunset colormap with white at 0 ---
sunset_rgb = [
    (247, 242, 229),  # light cream
    (236, 226, 192),  # light pink/yellow
    (227, 158, 62),   # orange
    (217, 84, 34),    # deep orange
    (157, 39, 46),    # red
    (30, 9, 31),      # dark purple/black
]

# Normalize RGB to [0, 1]
sunset_colors = [(r / 255, g / 255, b / 255) for r, g, b in sunset_rgb]

# 0.0 is White, 0.0001 starts the first color in sunset_colors
colors_with_white = [(1, 1, 1)] + sunset_colors
nodes = [0.0, 0.0001] + list(np.linspace(0.25, 1.0, len(sunset_colors) - 1))

sunset_cmap = LinearSegmentedColormap.from_list(
    "kSunset_white_min", list(zip(nodes, colors_with_white)), N=256
)


# --- 2. Updated Plotting Function ---
def plot_split_tpc_2d(
    df: pd.DataFrame,
    pos_type: str = "start",  # 'start' or 'end'
    weight_col: tuple = None,
    cmap: LinearSegmentedColormap = sunset_cmap,
    figsize: tuple = (15, 6),
):
    """Generates a 1x2 multiplot of Y (vertical) vs Z (horizontal) split by X < 0 (left TPC) and X >= 0 (right TPC)."""
    cols = start_cols if pos_type == "start" else end_cols

    # Safely extract column data
    try:
        x_raw = df[cols["x"]]
        y_raw = df[cols["y"]]
        z_raw = df[cols["z"]]
    except KeyError as e:
        raise KeyError(
            f"Could not find coordinate columns for pos_type='{pos_type}'. Expected column tuple: {e}"
        )

    # Filter out NaNs across spatial coordinates
    mask = x_raw.notna() & y_raw.notna() & z_raw.notna()
    plot_df = df[mask]

    x_vals = plot_df[cols["x"]].to_numpy(dtype=float)
    y_vals = plot_df[cols["y"]].to_numpy(dtype=float)
    z_vals = plot_df[cols["z"]].to_numpy(dtype=float)

    # Weights resolution
    if weight_col and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0).to_numpy(dtype=float)
    else:
        weights = np.ones_like(x_vals, dtype=float)

    # Keep strictly finite values
    finite_mask = (
        np.isfinite(x_vals)
        & np.isfinite(y_vals)
        & np.isfinite(z_vals)
        & np.isfinite(weights)
    )
    x_vals = x_vals[finite_mask]
    y_vals = y_vals[finite_mask]
    z_vals = z_vals[finite_mask]
    weights = weights[finite_mask]

    # Split into Left TPC (X < 0) and Right TPC (X >= 0)
    mask_neg_x = x_vals < 0
    mask_pos_x = x_vals >= 0

    # Calculate global max bin count across both TPCs
    h_left, _, _ = np.histogram2d(
        z_vals[mask_neg_x], y_vals[mask_neg_x], bins=[BINS_Z, BINS_Y], weights=weights[mask_neg_x]
    )
    h_right, _, _ = np.histogram2d(
        z_vals[mask_pos_x], y_vals[mask_pos_x], bins=[BINS_Z, BINS_Y], weights=weights[mask_pos_x]
    )

    vmax = max(h_left.max() if h_left.size > 0 else 0, h_right.max() if h_right.size > 0 else 0)
    vmax = vmax if vmax > 0 else 1.0

    # Setup 1x2 Subplots
    fig, (ax_left, ax_right) = plt.subplots(
        1, 2, figsize=figsize, sharey=True, gridspec_kw={"wspace": 0.08}
    )

    # --- Left Plot: X < 0 cm ---
    im0 = ax_left.hist2d(
        z_vals[mask_neg_x],
        y_vals[mask_neg_x],
        bins=[BINS_Z, BINS_Y],
        weights=weights[mask_neg_x],
        cmap=cmap,
        vmin=0,     # Bins with 0 entries stay white (node 0.0)
        vmax=vmax,
    )[3]

    ax_left.set_title(
        f"Track {pos_type.capitalize()}: $X < 0$ cm ({mask_neg_x.sum()} hits)",
        fontsize=14,
        pad=10,
    )
    ax_left.set_xlabel("Z [cm]", fontsize=14)
    ax_left.set_ylabel("Y [cm]", fontsize=14)
    ax_left.set_xlim(BINS_Z[0], BINS_Z[-1])
    ax_left.set_ylim(BINS_Y[0], BINS_Y[-1])
    ax_left.grid(alpha=0.3, linestyle="--")

    # --- Right Plot: X >= 0 cm ---
    im1 = ax_right.hist2d(
        z_vals[mask_pos_x],
        y_vals[mask_pos_x],
        bins=[BINS_Z, BINS_Y],
        weights=weights[mask_pos_x],
        cmap=cmap,
        vmin=0,     # Bins with 0 entries stay white (node 0.0)
        vmax=vmax,
    )[3]

    ax_right.set_title(
        f"Track {pos_type.capitalize()}: $X \\geq 0$ cm ({mask_pos_x.sum()} hits)",
        fontsize=14,
        pad=10,
    )
    ax_right.set_xlabel("Z [cm]", fontsize=14)
    ax_right.set_xlim(BINS_Z[0], BINS_Z[-1])
    ax_right.grid(alpha=0.3, linestyle="--")

    # Shared Colorbar
    cbar = fig.colorbar(im1, ax=[ax_left, ax_right], pad=0.02)
    cbar.set_label("Weighted Entries", fontsize=12)

    return fig, (ax_left, ax_right)

In [ ]:
weight_col = ('slc', 'wgt', '', '', '', '')  # Pass None if unweighted

# 1. Track START positions multiplot
fig_start, axes_start = plot_split_tpc_2d(
    df=mc_bnb_pfp_df,
    pos_type="start",
    weight_col=weight_col
)

# 2. Track END positions multiplot
fig_end, axes_end = plot_split_tpc_2d(
    df=mc_bnb_pfp_df,
    pos_type="end",
    weight_col=weight_col
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Column Tuples ---
start_x_col = ("pfp", "trk", "start", "x", "", "")
end_x_col   = ("pfp", "trk", "end", "x", "", "")
p_type_col  = ("pfp", "trk", "truth", "p", "p_type", "")
weight_col  = ("slc", "wgt", "", "", "", "")  # Set to None if unweighted

# --- Colors & LaTeX Labels ---
p_type_colors = {
    "muon": "#1f77b4",   # Blue
    "pion": "#d62728",   # Red
    "proton": "#2ca02c", # Green
    "other": "#7f7f7f",  # Grey
}

p_type_labels = {
    "muon": r"$\mu$",
    "pion": r"$\pi^\pm$",
    "proton": r"$p$",
    "other": "Other",
}

# --- Binning across the TPC X range [-200, 200] cm ---
bins_x = np.linspace(-200, 200, 81)

# 1. Track START X Histogram
fig_start, ax_start = plot_category_histogram(
    df=mc_bnb_pfp_df,
    var_col=start_x_col,
    category_col=p_type_col,
    weight_col=weight_col,
    bins=bins_x,
    category_colors=category_colors_pfp,
    category_labels=p_type_labels,
    xlabel=r"Track Start $X$ [cm]",
    ylabel="Weighted Events",
    title=r"Track Start $X$ Position by Particle Type",
    stacked=True,
)

# 2. Track END X Histogram
fig_end, ax_end = plot_category_histogram(
    df=mc_bnb_pfp_df,
    var_col=end_x_col,
    category_col=p_type_col,
    weight_col=weight_col,
    bins=bins_x,
    category_colors=category_colors_pfp,
    category_labels=p_type_labels,
    xlabel=r"Track End $X$ [cm]",
    ylabel="Weighted Events",
    title=r"Track End $X$ Position by Particle Type",
    stacked=True,
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Target Variables & Specific Binnings ---
variables_to_plot = [
    {
        "col": ("pfp", "trk", "chi2_exp_pol", "", "", ""),
        "xlabel": r"$\chi^2_{\mathrm{exp/pol}}$",
        "title": "",
        "bins": np.linspace(0.0, 1.0, 51),
        "vlines": (0.25, 0.75),  # 👈 Example vertical cut locations (or None)
    },
    {
        "col": ("pfp", "trk", "len", "", "", ""),
        "xlabel": r"Track Length [cm]",
        "title": "",
        "bins": np.linspace(0.0, 200.0, 51),
        "vlines": (-10.0, 100.0),  # 👈 Example vertical cut locations
    },
    {
        "col": ("pfp", "trk", "chi2pid", "best", "chi2_pion", ""),
        "xlabel": r"$\chi^2_{\pi}$",
        "title": "",
        "bins": np.linspace(0.0, 20.0, 51),
        "vlines": (-10.0, 6.0),  # 👈 Example vertical cut locations
    },
    {
        "col": ("pfp", "trk", "chi2pid", "best", "chi2_proton", ""),
        "xlabel": r"$\chi^2_{p}$",
        "title": "",
        "bins": np.linspace(0.0, 300.0, 61),
        "vlines": (-10.0, 130.0),  # 👈 Example vertical cut locations
    },
]


# --- Classification & Weighting Columns ---
p_type_col = ('pfp', 'trk', 'truth', 'p', 'p_type', '')
weight_col = ('slc', 'wgt', '', '', '', '')  # Pass None if unweighted

# --- Category Styling ---
p_type_colors = {
    "muon": "#1f77b4",   # Blue
    "pion": "#d62728",   # Red
    "proton": "#2ca02c", # Green
    "other": "#7f7f7f",  # Grey
}

p_type_labels = {
    "muon": r"$\mu$",
    "pion": r"$\pi^\pm$",
    "proton": r"$p$",
    "other": "Other",
}

# --- Loop and Plot ---
generated_figures = {}
from pathlib import Path

# Define output directory and ensure it exists
output_dir = Path("/exp/sbnd/data/users/lpelegri/TLEGraphs/SelectionGraphs")
output_dir.mkdir(parents=True, exist_ok=True)

for var in variables_to_plot:
    fig, ax = plot_category_histogram(
        df=mc_bnb_pfp_df,
        var_col=var["col"],
        category_col=p_type_col,
        weight_col=weight_col,
        bins=var["bins"],
        category_colors=category_colors_pfp,
        category_labels=p_type_labels,
        vlines=var.get("vlines", None),
        xlabel=var["xlabel"],
        ylabel="A.U.",
        title=var["title"],
        stacked=True,
    )
    
    # Store in memory dict
    var_key = var["col"]
    generated_figures[var_key] = (fig, ax)

    # Save figure to disk (cleans string to ensure a valid filename)
    filename_base = f"selection_{var_key}".replace("/", "_").replace(" ", "_")
    print(filename_base)
    # Save as PDF (for publications) and PNG (for quick previews)
    fig.savefig(output_dir / f"{filename_base}.pdf", bbox_inches="tight")

plt.show()